# Data load & Sanity Check

The model (`donderformer.models.taikonation`) and the train/validation loop (`donderformer.training`) live in `src/`. Evaluation, generation and plotting stay in this notebook for now; `notebooks/reference/baseline_cnn_lstm_full_pace.ipynb` is the original full-data PACE run.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from donderformer.data.feature_extractor import FeatureExtractor
from donderformer.data.taiko_dataset import Window
from donderformer.paths import DataPaths, outputs_dir
from donderformer.runs import new_run_dir

# Data root: DONDER_DATA env var, else <repo>/data. Use DataPaths.toy() for a quick local run.
paths = DataPaths()
SPLIT = "oni_v2"                              # folder under splits/, made by 01_make_split
extractor = FeatureExtractor(frame_ms=23)     # cache made by 02_extract_features

# Window, in 23 ms frames, around the first predicted frame i. Defaults are the paper's:
# 16 audio frames ending at i, the 15 previous notes, predict i .. i+3.
AUDIO_FRAMES = 16    # audio frames per window
HISTORY = 15         # previous note frames fed back in (<= AUDIO_FRAMES - 1 - LOOK_FORWARD)
LOOK_FORWARD = 0     # audio frames after i; 3 lets the audio cover all 4 predicted frames
WINDOW = Window(audio_frames=AUDIO_FRAMES, history=HISTORY, look_forward=LOOK_FORWARD)
print("Window:", WINDOW.describe())

DROPOUT = 0.2        # drop probability; the released code's keep_prob=0.8

MODEL_NAME = "taikonation"
ALIAS = WINDOW.name                           # Something descriptive, such as Taikonation_baseline, etc. or something about the windown length

# Every run gets its own folder: outputs/runs/<model>/<datetime>[-<alias>]-<split>-<extractor>/
RUN_DIR = new_run_dir(MODEL_NAME, SPLIT, extractor.id, alias=ALIAS)
print("Run folder:", RUN_DIR)
DATA_INFO = {"split": SPLIT, "features": extractor.id, "window": WINDOW.to_dict()}

# Training. Defaults are the full-data PACE run; TaikoNation's paper fine-tunes at batch size 1.
BATCH_SIZE = 128
EPOCHS = 10
LR = 1e-5
FINETUNE_BATCH_SIZE = 128
FINETUNE_EPOCHS = 2
FINETUNE_LR = 5e-6
HPARAMS = dict(batch_size=BATCH_SIZE, epochs=EPOCHS, lr=LR, finetune_batch_size=FINETUNE_BATCH_SIZE,
               finetune_epochs=FINETUNE_EPOCHS, finetune_lr=FINETUNE_LR)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# local setup
# device = torch.device(
#     "mps" if torch.backends.mps.is_available()
#     else "cuda" if torch.cuda.is_available()
#     else "cpu"
# )

# print("Device:", device)

# PACE - cuda expected
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("Device:", device)


if device.type == "cuda":
    p = torch.cuda.get_device_properties(device)
    print(f"{torch.cuda.get_device_name(device)} | {p.total_memory / 1e9:.0f} GB | {p.multi_processor_count} SMs | compute {p.major}.{p.minor}")
    print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| cuDNN", torch.backends.cudnn.version(), "| GPUs visible:", torch.cuda.device_count())
    print(f"in use by this kernel: {torch.cuda.memory_allocated() / 1e9:.2f} GB (peak {torch.cuda.max_memory_allocated() / 1e9:.2f} GB)")

### Model Pipeline

**One input window**  
↓  
**`AUDIO_FRAMES` audio frames (A × 80) + `HISTORY` previous note frames (H × 7)**  
↓  
**CNN extracts features from the audio**  
↓  
**Audio features are combined with previous note information**  
↓  
**LSTM learns temporal/rhythmic patterns**  
↓  
**Predict 4 timesteps (4 × 7 classes)**

In [ ]:
# Defined in src/donderformer/models/taikonation.py
from donderformer.models.taikonation import TaikoNationBaseline

# sanity check

In [ ]:
model = TaikoNationBaseline(WINDOW, dropout=DROPOUT).to(device)
from donderformer.data.taiko_dataset import TaikoDataset
sample_ds = TaikoDataset(paths, SPLIT, extractor, "train", WINDOW)
X, _ = sample_ds.chart_windows(sample_ds.chart_ids[0])
sample = torch.from_numpy(X[:8]).to(device)
with torch.no_grad():
    output = model(sample)
print(WINDOW.describe())
print("Input:", sample.shape, "Output:", output.shape)
assert sample.shape[1] == model.n_inputs
assert output.shape == (sample.shape[0], WINDOW.targets, 7)

# data loader

Songs are loaded one at a time (see `donderformer.data.taiko_dataset`) because the full set is ~2k charts.

In [ ]:
from torch.utils.data import DataLoader
from donderformer.data.taiko_dataset import SongBatchSampler, TaikoDataset

train_dataset = TaikoDataset(paths, SPLIT, extractor, "train", WINDOW)
val_dataset = TaikoDataset(paths, SPLIT, extractor, "validation", WINDOW)
test_dataset = TaikoDataset(paths, SPLIT, extractor, "test", WINDOW)
train_loader = DataLoader(train_dataset,
    batch_sampler=SongBatchSampler(train_dataset, BATCH_SIZE, shuffle=True), num_workers=0)
val_loader = DataLoader(val_dataset,
    batch_sampler=SongBatchSampler(val_dataset, BATCH_SIZE), num_workers=0)
for name, ds in (("train", train_dataset), ("validation", val_dataset), ("test", test_dataset)):
    print(f"{name}: {len(ds.chart_ids)} charts, {len(ds)} windows")
print("Train batches:", len(train_loader), "Val batches:", len(val_loader))

# loss + optimizer

In [ ]:
# measures how wrong the model's predictions are compared withthe correct note classes
criterion = nn.CrossEntropyLoss()

# changes the model's weights to reduce the loss; Adam is a popular optimizer that works well in many situations
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR
)

# training & validation functions

In [ ]:
# predict -> calculate loss -> backpropagate -> update weights
# Defined in src/donderformer/training.py
from donderformer.training import evaluate, train_epoch

# Train (stage 1)

In [ ]:
import time
start_time = time.time()

train_losses = []
val_losses = []

for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss = evaluate(model, val_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(
        f"Epoch {epoch + 1:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Elapsed time: {(time.time() - start_time) / 60:.1f} min"
    )

# Second stage - fine tuning

TaikoNation fine-tunes with batch size 1 and learning rate 5e-6. The full-data PACE run used `FINETUNE_BATCH_SIZE = 128` for 2 epochs, since batch 1 over ~7M windows takes days; set it to 1 in the config cell to replicate the paper.

In [ ]:
train_loader_finetune = DataLoader(
    train_dataset,
    batch_sampler=SongBatchSampler(train_dataset, FINETUNE_BATCH_SIZE, shuffle=True),
    num_workers=0
)
optimizer = torch.optim.Adam(model.parameters(), lr=FINETUNE_LR)

finetune_train_losses = []
finetune_val_losses = []

for epoch in range(FINETUNE_EPOCHS):
    train_loss = train_epoch(model, train_loader_finetune, optimizer, criterion, device)
    val_loss = evaluate(model, val_loader, criterion, device)
    finetune_train_losses.append(train_loss)
    finetune_val_losses.append(val_loss)
    print(
        f"Fine-tune Epoch {epoch + 1:02d} | "
        f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
        f"Elapsed time: {(time.time() - start_time) / 60:.1f} min"
    )

# save model

In [ ]:
from donderformer.runs import save_run

checkpoint = save_run(
    RUN_DIR,
    model,
    config={"model": MODEL_NAME, "alias": ALIAS, "data": DATA_INFO, "dropout": DROPOUT,
            "hparams": HPARAMS, "device": str(device)},
    losses={
        "train": train_losses,
        "val": val_losses,
        "finetune_train": finetune_train_losses,
        "finetune_val": finetune_val_losses,
    },
)

print("Saved", checkpoint)

# Make predictions on the test song

In [ ]:
# Evaluation writes into the run's eval/ folder.
# To evaluate an earlier run instead of the model just trained, run the cells up to the data loader
# (same SPLIT and extractor as that run), then uncomment these lines:
# from donderformer.runs import load_run
# RUN_DIR = outputs_dir() / "runs" / MODEL_NAME / "<run folder name>"
# load_run(RUN_DIR, model, map_location=device)

EVAL_DIR = RUN_DIR / "eval"
TF_DIR = EVAL_DIR / "teacher_forced"    # teacher-forced test metrics
PLOTS_DIR = EVAL_DIR / "plots"
for d in (TF_DIR, PLOTS_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("Evaluating", RUN_DIR.name)

### Test Prediction Pipeline

**Trained Model + Test Audio/Note Windows**  
↓  
**Model Prediction**  
↓  
**Raw Scores (Logits)**  
↓  
**Softmax**  
↓  
**Probabilities for Each of the 7 Note Classes**  
↓  
**Used for Chart Generation**

In [ ]:
# Inference is now performed song-by-song in Cell 31, in small GPU batches.
model.eval()
INFERENCE_BATCH_SIZE = 64
print("Ready to evaluate", len(test_dataset.chart_ids), "test charts")

# Combining the overlapping predictions from different windows to produce one final 7-class probability distribution for each 23 ms timestep.

In [ ]:
def aggregate_overlapping_predictions(probs):
    # probs: (number_of_windows, 4, 7), from ONE song only
    n = len(probs)
    if n == 0:
        return np.empty((0, 7), dtype=np.float32)
    aggregated = np.zeros((n + 3, 7), dtype=np.float32)
    counts = np.zeros(n + 3, dtype=np.float32)
    for j in range(4):
        aggregated[j:j+n] += probs[:, j, :]
        counts[j:j+n] += 1
    return aggregated / counts[:, None]

# Generating chart

Takes the final probability distribution for every 23 ms timestep and turns it into an actual generated chart.

In [ ]:
CLASS_NAMES = ["none", "don", "ka", "DON", "KA", "drumroll", "denden"]
rng = np.random.default_rng(42)

def sample_chart(aggregated):
    return np.array([rng.choice(7, p=p / p.sum()) for p in aggregated],
                    dtype=np.int64)

# Target vs. Model generated

In [ ]:
def reconstruct_true_sequence(Y):
    # Y: (number_of_windows, 4, 7), for ONE song only
    targets = Y.argmax(axis=2)
    n = len(targets)
    if n == 0:
        return np.empty(0, dtype=np.int64)
    sequence = np.empty(n + 3, dtype=np.int64)
    sequence[:n] = targets[:, 0]
    sequence[n:] = targets[-1, 1:4]
    return sequence

# Onset evaluation

Regardless of note type like Don and Ka, this measures generated notes occur near the ground-truth note timings (within +- 23ms).

In [ ]:
def onset_metrics(true_seq, pred_seq, tolerance=1):
    true_onsets = np.where(true_seq != 0)[0]
    pred_onsets = np.where(pred_seq != 0)[0]

    matched_true = set()
    matches = 0

    for pred in pred_onsets:
        candidates = [
            t for t in true_onsets
            if abs(t - pred) <= tolerance and t not in matched_true
        ]

        if candidates:
            best = min(candidates, key=lambda t: abs(t - pred))
            matched_true.add(best)
            matches += 1

    precision = matches / len(pred_onsets) if len(pred_onsets) else 0
    recall = matches / len(true_onsets) if len(true_onsets) else 0

    if precision + recall:
        f1 = 2 * precision * recall / (precision + recall)
    else:
        f1 = 0

    return precision, recall, f1, matches

# Onset + type eval

In [ ]:
def joint_onset_type_metrics(true_seq, pred_seq, tolerance=1):
    true_onsets = np.where(true_seq != 0)[0]
    pred_onsets = np.where(pred_seq != 0)[0]

    matched_true = set()
    matches = 0

    for pred in pred_onsets:
        candidates = [
            t for t in true_onsets
            if abs(t - pred) <= tolerance
            and true_seq[t] == pred_seq[pred]
            and t not in matched_true
        ]

        if candidates:
            best = min(candidates, key=lambda t: abs(t - pred))
            matched_true.add(best)
            matches += 1

    precision = matches / len(pred_onsets) if len(pred_onsets) else 0
    recall = matches / len(true_onsets) if len(true_onsets) else 0

    if precision + recall:
        f1 = 2 * precision * recall / (precision + recall)
    else:
        f1 = 0

    return precision, recall, f1, matches

def score_from_counts(tp, predicted, actual):
    precision = tp / predicted if predicted else 0.0
    recall = tp / actual if actual else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1

# Micro-averaged evaluation: accumulate counts across songs, never across boundaries.
model.eval()
total_true = total_pred = total_onset_matches = total_joint_matches = 0
true_class_counts = np.zeros(7, dtype=np.int64)
pred_class_counts = np.zeros(7, dtype=np.int64)

for song_index, song_id in enumerate(test_dataset.chart_ids, start=1):
    X, Y = test_dataset.chart_windows(song_id)
    if len(X) == 0:
        continue
    song_probs = []
    with torch.no_grad():
        for start in range(0, len(X), INFERENCE_BATCH_SIZE):
            batch = torch.from_numpy(
                X[start:start + INFERENCE_BATCH_SIZE].astype(np.float32)
            ).to(device)
            logits = model(batch)
            song_probs.append(torch.softmax(logits, dim=2).cpu().numpy())
    probs = np.concatenate(song_probs, axis=0)
    generated = sample_chart(aggregate_overlapping_predictions(probs))
    true_sequence = reconstruct_true_sequence(Y)
    assert len(generated) == len(true_sequence)

    _, _, _, onset_matches = onset_metrics(true_sequence, generated, tolerance=1)
    _, _, _, joint_matches = joint_onset_type_metrics(true_sequence, generated, tolerance=1)
    total_onset_matches += onset_matches
    total_joint_matches += joint_matches
    total_true += np.count_nonzero(true_sequence)
    total_pred += np.count_nonzero(generated)
    true_class_counts += np.bincount(true_sequence, minlength=7)
    pred_class_counts += np.bincount(generated, minlength=7)
    if song_index % 20 == 0:
        print(f"Evaluated {song_index}/{len(test_dataset.chart_ids)} charts")

precision, recall, f1 = score_from_counts(total_onset_matches, total_pred, total_true)
precision_type, recall_type, f1_type = score_from_counts(total_joint_matches, total_pred, total_true)
print(f"Onset: precision={precision:.4f}, recall={recall:.4f}, F1={f1:.4f}")
print(f"Onset + type: precision={precision_type:.4f}, recall={recall_type:.4f}, F1={f1_type:.4f}")
for name, actual, predicted in zip(CLASS_NAMES, true_class_counts, pred_class_counts):
    print(f"{name:10s} true={actual:8d} generated={predicted:8d}")

In [ ]:
results = {
    "onset_precision": float(precision),
    "onset_recall": float(recall),
    "onset_f1": float(f1),
    "joint_precision": float(precision_type),
    "joint_recall": float(recall_type),
    "joint_f1": float(f1_type)
}

import json

# Saved in the run's eval/teacher_forced/ folder
(TF_DIR / "results.json").write_text(json.dumps(results, indent=2))

print("Evaluation results saved to", TF_DIR / "results.json")
print(results)

# OCHuman-style Evaluation (±1 timestep)

In [ ]:
# Uses the already-trained model

import numpy as np
import torch

def ochuman_score(true_seq, pred_seq, tolerance=1):
    true_onsets = np.flatnonzero(true_seq != 0)
    pred_mask = pred_seq != 0

    if len(true_onsets) == 0:
        return 0, 0

    matched = 0
    for t in true_onsets:
        left = max(0, t - tolerance)
        right = min(len(pred_seq), t + tolerance + 1)

        if np.any(pred_mask[left:right]):
            matched += 1

    return matched, len(true_onsets)


model.eval()

total_matched = 0
total_true = 0
song_scores = []

for song_index, song_id in enumerate(test_dataset.chart_ids, start=1):

    X, Y = test_dataset.chart_windows(song_id)

    if len(X) == 0:
        continue

    song_probs = []

    with torch.no_grad():
        for start in range(0, len(X), INFERENCE_BATCH_SIZE):
            batch = torch.from_numpy(
                X[start:start + INFERENCE_BATCH_SIZE].astype(np.float32)
            ).to(device)

            logits = model(batch)
            probs = torch.softmax(logits, dim=2)
            song_probs.append(probs.cpu().numpy())

    probs = np.concatenate(song_probs, axis=0)

    generated = sample_chart(
        aggregate_overlapping_predictions(probs)
    )
    true_sequence = reconstruct_true_sequence(Y)

    assert len(generated) == len(true_sequence)

    matched, actual = ochuman_score(
        true_sequence,
        generated,
        tolerance=1
    )

    total_matched += matched
    total_true += actual

    if actual > 0:
        song_scores.append(matched / actual)

    if song_index % 20 == 0:
        print(f"Evaluated {song_index}/{len(test_dataset.chart_ids)} charts")


# Overall score (micro-average)
ochuman_micro = total_matched / total_true if total_true else 0

# Per-chart average (macro-average)
ochuman_macro = np.mean(song_scores) if song_scores else 0

print("\n===== OCHuman-style Results =====")
print(f"Micro-average: {ochuman_micro:.4f} ({ochuman_micro:.2%})")
print(f"Macro-average: {ochuman_macro:.4f} ({ochuman_macro:.2%})")
print(f"Charts evaluated: {len(song_scores)}")

# Save results separately
np.savez(
    TF_DIR / "ochuman.npz",
    micro=ochuman_micro,
    macro=ochuman_macro,
    song_scores=np.array(song_scores)
)

print("Results saved to", TF_DIR / "ochuman.npz")

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

# Select a test song
song_id = test_dataset.chart_ids[0]
X, Y = test_dataset.chart_windows(song_id)

# Generate predictions
model.eval()
song_probs = []

with torch.no_grad():
    for start in range(0, len(X), INFERENCE_BATCH_SIZE):
        batch = torch.from_numpy(
            X[start:start + INFERENCE_BATCH_SIZE].astype(np.float32)
        ).to(device)

        logits = model(batch)
        song_probs.append(
            torch.softmax(logits, dim=2).cpu().numpy()
        )

probs = np.concatenate(song_probs, axis=0)

generated = sample_chart(
    aggregate_overlapping_predictions(probs)
)
true_sequence = reconstruct_true_sequence(Y)

assert len(generated) == len(true_sequence)

# Plot the first 300 timesteps (~6.9 seconds if 23ms per step)
start_step = 0
num_steps = 300
end_step = min(start_step + num_steps, len(true_sequence))

colors = {
    1: "red",       # don
    2: "blue",      # ka
    3: "darkred",   # DON
    4: "darkblue",  # KA
    5: "orange",    # drumroll
    6: "purple"     # denden
}

fig, axes = plt.subplots(2, 1, figsize=(16, 5), sharex=True)

for ax, seq, title in zip(
    axes,
    [true_sequence, generated],
    ["Human (Ground Truth)", "AI Generated"]
):
    for note_type, color in colors.items():
        positions = np.where(
            seq[start_step:end_step] == note_type
        )[0] + start_step

        ax.scatter(
            positions,
            np.full(len(positions), 1),
            color=color,
            s=55,
            label=CLASS_NAMES[note_type]
        )

    ax.set_title(title)
    ax.set_ylim(0.5, 1.5)
    ax.set_yticks([])
    ax.grid(axis="x", alpha=0.25)

axes[0].legend(
    loc="upper center",
    bbox_to_anchor=(0.5, 1.5),
    ncol=6,
    fontsize=9
)

axes[-1].set_xlabel("Timestep (1 step = 23ms)")

# autoregressive generation

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

def generate_autoregressive(model, X, device, seed=42):
    """
    Autoregressive Taiko chart generation.

    X: preprocessed windows, shape (N, A*80 + H*7); only the audio part is used.
    Ground-truth note history is NEVER used.
    """
    history_frames = model.window.history
    model.eval()
    rng = np.random.default_rng(seed)

    n = len(X)
    total_steps = n + 3

    generated = np.zeros(total_steps, dtype=np.int64)

    # Accumulate overlapping predictions
    prob_sums = np.zeros((total_steps, 7), dtype=np.float32)
    counts = np.zeros(total_steps, dtype=np.int32)

    eye = np.eye(7, dtype=np.float32)

    # Keep only audio; discard ground-truth note history
    audio = X[:, :X.shape[1] - history_frames * 7].astype(np.float32)

    with torch.inference_mode():
        for t in range(total_steps):

            if t < n:
                # Previous GENERATED notes
                history = np.zeros((history_frames, 7), dtype=np.float32)

                start = max(0, t - history_frames)
                previous = generated[start:t]

                if len(previous) > 0:
                    history[-len(previous):] = eye[previous]

                # Construct model input
                x = np.concatenate([
                    audio[t],
                    history.reshape(-1)
                ])

                batch = torch.from_numpy(
                    x[None, :]
                ).to(device)

                logits = model(batch)

                probs = torch.softmax(
                    logits, dim=2
                )[0].cpu().numpy()

                # Predictions for t, t+1, t+2, t+3
                for j in range(4):
                    idx = t + j
                    prob_sums[idx] += probs[j]
                    counts[idx] += 1

            # Sample current timestep using predictions
            # available so far (no future-note leakage)
            if counts[t] > 0:
                p = prob_sums[t] / counts[t]
                p = p / p.sum()
                generated[t] = rng.choice(7, p=p)

    return generated


# ========================================
# Evaluate one test song
# ========================================

song_id = test_dataset.chart_ids[0]
X, Y = test_dataset.chart_windows(song_id)

true_sequence = reconstruct_true_sequence(Y)

generated_ar = generate_autoregressive(
    model, X, device, seed=42
)

assert len(generated_ar) == len(true_sequence)

# Calculate metrics
p, r, f1, _ = onset_metrics(
    true_sequence,
    generated_ar,
    tolerance=1
)

jp, jr, jf1, _ = joint_onset_type_metrics(
    true_sequence,
    generated_ar,
    tolerance=1
)

print("===== Autoregressive Results =====")
print("Song:", song_id)
print(f"Onset Precision: {p:.4f}")
print(f"Onset Recall:    {r:.4f}")
print(f"Onset F1:        {f1:.4f}")
print(f"Joint F1:        {jf1:.4f}")

print("True notes:", np.count_nonzero(true_sequence))
print("Generated notes:", np.count_nonzero(generated_ar))

# ========================================
# Visualize first 300 timesteps
# ========================================

start_step = 0
end_step = min(300, len(true_sequence))

colors = {
    1: "red",
    2: "blue",
    3: "darkred",
    4: "darkblue",
    5: "orange",
    6: "purple"
}

fig, axes = plt.subplots(
    2, 1, figsize=(16, 5), sharex=True
)

for ax, seq, title in zip(
    axes,
    [true_sequence, generated_ar],
    ["Human (Ground Truth)", "AI (Autoregressive)"]
):
    for note_type, color in colors.items():
        positions = np.where(
            seq[start_step:end_step] == note_type
        )[0] + start_step

        ax.scatter(
            positions,
            np.ones(len(positions)),
            color=color,
            s=55,
            label=CLASS_NAMES[note_type]
        )

    ax.set_title(title)
    ax.set_ylim(0.5, 1.5)
    ax.set_yticks([])
    ax.grid(axis="x", alpha=0.25)

axes[0].legend(
    loc="upper center",
    bbox_to_anchor=(0.5, 1.5),
    ncol=6
)

axes[-1].set_xlabel("Timestep (1 step = 23ms)")

plt.tight_layout()
plt.savefig(
    PLOTS_DIR / "human_vs_autoregressive.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()

In [ ]:

import numpy as np
import csv
import torch
import json
import time
from pathlib import Path

# ==========================================
# 1. Configuration
# ==========================================

SAVE_DIR = EVAL_DIR / "ar_test"          # metrics: results.json, summary.csv
CHARTS_DIR = SAVE_DIR / "charts"         # generated sequences: <chart_id>.npz
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

test_songs = test_dataset.chart_ids

# ==========================================
# 2. Helper functions
# ==========================================

def remove_adjacent_notes(sequence):
    cleaned = sequence.copy()

    for t in range(1, len(sequence)):
        if sequence[t - 1] != 0 and sequence[t] != 0:
            cleaned[t] = 0

    return cleaned


def to_binary_chart(sequence):
    sequence = np.asarray(sequence)

    # Discrete notes
    binary = np.isin(sequence, [1, 2, 3, 4])

    # Drumroll / Denden: only count onset
    held = np.isin(sequence, [5, 6])
    starts = np.r_[True, sequence[1:] != sequence[:-1]]

    binary |= held & starts
    return binary.astype(np.uint8)


def paper_ochuman(human, ai):
    """
    Paper-style approximation:
    Exact binary agreement, with ±1 timestep
    tolerance for human note onsets.
    """
    human = np.asarray(human, dtype=bool)
    ai = np.asarray(ai, dtype=bool)

    assert len(human) == len(ai)

    correct = (human == ai)

    nearby = ai.copy()
    nearby[1:] |= ai[:-1]
    nearby[:-1] |= ai[1:]

    correct |= human & nearby

    return float(correct.mean())


def extract_patterns(binary_chart, window=8):
    chart = np.asarray(binary_chart, dtype=np.uint8)

    return {
        tuple(chart[i:i + window])
        for i in range(len(chart) - window + 1)
    }


def calculate_paper_metrics(human, generated):
    h = to_binary_chart(human)
    a = to_binary_chart(generated)

    dc = float(np.mean(h == a) * 100)
    oc = paper_ochuman(h, a) * 100

    hp = extract_patterns(h)
    ap = extract_patterns(a)

    over = len(ap) / 256 * 100
    hi = len(hp & ap) / len(hp) * 100 if hp else 0.0

    return {
        "DCHuman": dc,
        "OCHuman": oc,
        "Over_PSpace": over,
        "HI_PSpace": hi,
        "human_onsets": int(h.sum()),
        "ai_onsets": int(a.sum())
    }


def calculate_f1_metrics(human, predicted):
    p, r, f1, onset_matches = onset_metrics(
        human, predicted, tolerance=1
    )

    jp, jr, jf1, joint_matches = joint_onset_type_metrics(
        human, predicted, tolerance=1
    )

    return {
        "onset_precision": float(p),
        "onset_recall": float(r),
        "onset_f1": float(f1),
        "joint_precision": float(jp),
        "joint_recall": float(jr),
        "joint_f1": float(jf1),
        "onset_matches": int(onset_matches),
        "joint_matches": int(joint_matches)
    }


# ==========================================
# 3. Evaluate every test song
# ==========================================

model.eval()

all_results = []
start_time = time.time()

for song_index, song_id in enumerate(test_songs, start=1):

    save_path = CHARTS_DIR / f"{song_id}.npz"

    # Resume from previously saved sequences
    if save_path.exists():
        with np.load(save_path) as data:
            true_seq = data["human"]
            generated = data["generated"]
            generated_clean = data["clean"]
    else:
        X, Y = test_dataset.chart_windows(song_id)

        if len(X) == 0:
            continue

        true_seq = reconstruct_true_sequence(Y)

        # Same seed as original evaluation
        generated = generate_autoregressive(
            model, X, device, seed=42 + song_index
        )

        assert len(generated) == len(true_seq)

        generated_clean = remove_adjacent_notes(generated)

        # Save all sequences immediately
        np.savez_compressed(
            save_path,
            human=true_seq,
            generated=generated,
            clean=generated_clean
        )

    # Calculate both types of metrics
    result = {"song_id": str(song_id)}

    for label, predicted in [
        ("raw", generated),
        ("clean", generated_clean)
    ]:
        result[label] = {
            **calculate_f1_metrics(true_seq, predicted),
            **calculate_paper_metrics(true_seq, predicted)
        }

    all_results.append(result)

    # Save progress after every song
    with open(SAVE_DIR / "results.json", "w") as f:
        json.dump(all_results, f, indent=2)

    if song_index % 10 == 0 or song_index == len(test_songs):
        elapsed = (time.time() - start_time) / 60

        print(
            f"Evaluated {song_index}/{len(test_songs)} "
            f"| Elapsed: {elapsed:.1f} min",
            flush=True
        )


# ==========================================
# 4. Aggregate results
# ==========================================

print("\n========== FINAL TEST RESULTS ==========")
print(f"Songs evaluated: {len(all_results)}")

summary = []

for label in ["raw", "clean"]:
    print(f"\n========== {label.upper()} ==========")

    metrics = [
        "onset_precision",
        "onset_recall",
        "onset_f1",
        "joint_precision",
        "joint_recall",
        "joint_f1",
        "DCHuman",
        "OCHuman",
        "Over_PSpace",
        "HI_PSpace"
    ]

    for metric in metrics:
        values = [r[label][metric] for r in all_results]
        mean = float(np.mean(values))

        if metric in ["DCHuman", "OCHuman",
                      "Over_PSpace", "HI_PSpace"]:
            print(f"{metric:20s}: {mean:.3f}%")
        else:
            print(f"{metric:20s}: {mean:.4f}")

        summary.append({
            "version": label,
            "metric": metric,
            "mean": mean
        })

with open(SAVE_DIR / "summary.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["version", "metric", "mean"])
    writer.writeheader()
    writer.writerows(summary)

print(
    f"\nTotal time: {(time.time()-start_time)/60:.2f} min"
)
print(f"Metrics saved in: {SAVE_DIR.resolve()}")
print(f"Sequences saved in: {CHARTS_DIR.resolve()}")


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.patches import Patch

CHARTS_DIR = EVAL_DIR / "ar_test" / "charts"

# First test song
song_id = str(test_dataset.chart_ids[0])

with np.load(CHARTS_DIR / f"{song_id}.npz") as data:
    human = data["human"]
    raw = data["generated"]
    clean = data["clean"]

# Display first 20 seconds
STEP_SECONDS = 0.023
START_SEC = 0
END_SEC = 20

start = int(START_SEC / STEP_SECONDS)
end = min(int(END_SEC / STEP_SECONDS), len(human))

NOTE_NAMES = {
    1: "Don",
    2: "Ka",
    3: "Big Don",
    4: "Big Ka",
    5: "Drumroll",
    6: "Denden"
}

NOTE_COLORS = {
    1: "red",
    2: "royalblue",
    3: "darkred",
    4: "navy",
    5: "orange",
    6: "purple"
}

fig, axes = plt.subplots(
    3, 1,
    figsize=(17, 7),
    sharex=True,
    constrained_layout=True
)

for ax, (label, seq) in zip(axes, [
    ("Human (Ground Truth)", human),
    ("AI - RAW", raw),
    ("AI - CLEAN", clean)
]):
    # Show only discrete hits and beginnings of held notes
    for note_type in range(1, 7):
        positions = np.flatnonzero(seq[start:end] == note_type) + start

        if note_type in [5, 6]:
            positions = positions[
                (positions == 0) |
                (seq[np.maximum(positions - 1, 0)] != note_type)
            ]

        times = positions * STEP_SECONDS

        ax.scatter(
            times,
            np.zeros(len(times)),
            c=NOTE_COLORS[note_type],
            s=65 if note_type <= 2 else 95,
            marker="o" if note_type <= 4 else "D",
            alpha=0.85
        )

    ax.set_title(label, fontsize=13, fontweight="bold")
    ax.set_ylim(-0.7, 0.7)
    ax.set_yticks([])
    ax.grid(axis="x", alpha=0.25)

axes[-1].set_xlabel("Time (seconds)")

legend = [
    Patch(color=NOTE_COLORS[k], label=NOTE_NAMES[k])
    for k in range(1, 7)
]

fig.legend(
    handles=legend,
    loc="upper center",
    ncol=6,
    bbox_to_anchor=(0.5, 1.05)
)

fig.suptitle(
    f"Taiko Chart Comparison: {song_id}",
    fontsize=16,
    fontweight="bold"
)

plt.savefig(
    PLOTS_DIR / "taiko_chart_comparison.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

print(f"Song: {song_id}")
print(f"Displayed: {START_SEC}-{END_SEC} seconds")
print("Saved:", PLOTS_DIR / "taiko_chart_comparison.png")
